# 实践项目：中文新闻分类系统



## 1. 学习目标

完成本课程后，你应该能够：

- 独立完成一个完整的文本分类项目
- 处理真实的中文新闻数据
- 对比不同算法的效果
- 优化模型性能
- 将模型应用到实际场景

## 2. 项目介绍

### 2.1 项目目标

构建一个中文新闻分类系统，能够自动将新闻分为以下5个类别：

- 科技
- 体育
- 娱乐
- 财经
- 社会


### 2.2 数据集介绍


- 数据规模：50000条新闻
- 类别数量：5类
- 数据路径：`/home/jovyan/work/datasets/5fbcdfa05005208e83d1ede4-momodel/news`

## 3. 步骤1：加载数据

首先，我们需要加载新闻数据集。

In [ ]:
import os

# 数据集路径
dataset_path = "/home/jovyan/work/datasets/5fbcdfa05005208e83d1ede4-momodel/news"

# 创建空列表存储文本和标签
texts = []
labels = []

# 获取目录下所有文件
files = os.listdir(dataset_path)

print("开始加载数据...")

# 遍历每个文件
for file in files:
    path = os.path.join(dataset_path, file)
    # 跳过隐藏文件和文件夹
    if not os.path.isdir(path) and not file[0] == '.':
        # 获取类别名（去掉.txt后缀）
        category = file[:-4]
        # 打开文件，使用UTF-8编码读取中文
        with open(path, 'r', encoding='UTF-8') as f:
            # 读取每一行新闻
            for line in f.readlines():
                line = line.strip()  # 去除首尾空格
                if line:  # 跳过空行
                    texts.append(line)
                    labels.append(category)

print(f"✅ 数据加载完成！")
print(f"总数据量: {len(texts)}")
print(f"\n各类别数量：")
from collections import Counter
label_counts = Counter(labels)
for label, count in label_counts.items():
    print(f"  {label}: {count}")

### 3.1 查看样本数据

In [ ]:
# 查看每个类别的一条样本
print("样本数据展示：\n")
print("="*80)

shown_labels = set()
for text, label in zip(texts, labels):
    if label not in shown_labels:
        print(f"\n类别: {label}")
        print(f"内容: {text[:100]}...")  # 只显示前100个字符
        print("-"*80)
        shown_labels.add(label)
        if len(shown_labels) == 5:
            break

## 4. 步骤2：数据预处理

对新闻文本进行分词处理。

In [ ]:
import jieba

print("开始分词处理...")

# 对所有文本进行分词
texts_segmented = []
for i, text in enumerate(texts):
    # 使用jieba分词
    words = jieba.cut(text)
    # 用空格连接分词结果
    text_seg = " ".join(words)
    texts_segmented.append(text_seg)
    
    # 显示进度
    if (i + 1) % 10000 == 0:
        print(f"已处理 {i + 1}/{len(texts)} 条")

print(f"\n✅ 分词完成！")
print(f"\n分词示例：")
print(f"原文: {texts[0][:50]}...")
print(f"分词: {texts_segmented[0][:50]}...")

## 5. 步骤3：划分训练集和测试集

In [ ]:
from sklearn.model_selection import train_test_split

# 划分数据集（80%训练，20%测试）
X_train, X_test, y_train, y_test = train_test_split(
    texts_segmented, 
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels  # 保持各类别比例一致
)

print(f"数据集划分完成：")
print(f"训练集大小: {len(X_train)}")
print(f"测试集大小: {len(X_test)}")
print(f"\n训练集类别分布：")
train_counts = Counter(y_train)
for label, count in train_counts.items():
    print(f"  {label}: {count}")

## 6. 步骤4：特征提取

使用TF-IDF将文本转换为数字向量。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("开始特征提取...")

# 创建TF-IDF向量化器
vectorizer = TfidfVectorizer(
    max_features=5000,  # 只保留最重要的5000个特征
    min_df=5,           # 至少在5个文档中出现
    max_df=0.8          # 最多在80%的文档中出现
)

# 拟合并转换训练集
X_train_vec = vectorizer.fit_transform(X_train)

# 转换测试集
X_test_vec = vectorizer.transform(X_test)

print(f"✅ 特征提取完成！")
print(f"特征维度: {X_train_vec.shape[1]}")
print(f"训练集形状: {X_train_vec.shape}")
print(f"测试集形状: {X_test_vec.shape}")

## 7. 步骤5：训练模型

我们将尝试多个分类算法，对比它们的效果。

### 7.1 模型1：朴素贝叶斯

In [ ]:
from sklearn.naive_bayes import MultinomialNB
import time

print("训练朴素贝叶斯模型...")
start_time = time.time()

# 创建并训练模型
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_vec, y_train)

# 预测
y_pred_nb = nb_classifier.predict(X_test_vec)

train_time = time.time() - start_time
print(f"✅ 训练完成！耗时: {train_time:.2f}秒")

### 7.2 模型2：逻辑回归

In [ ]:
from sklearn.linear_model import LogisticRegression

print("训练逻辑回归模型...")
start_time = time.time()

# 创建并训练模型
lr_classifier = LogisticRegression(max_iter=1000, random_state=42)
lr_classifier.fit(X_train_vec, y_train)

# 预测
y_pred_lr = lr_classifier.predict(X_test_vec)

train_time = time.time() - start_time
print(f"✅ 训练完成！耗时: {train_time:.2f}秒")

### 7.3 模型3：支持向量机

In [ ]:
from sklearn.svm import LinearSVC

print("训练SVM模型...")
start_time = time.time()

# 创建并训练模型（使用LinearSVC，速度更快）
svm_classifier = LinearSVC(random_state=42, max_iter=1000)
svm_classifier.fit(X_train_vec, y_train)

# 预测
y_pred_svm = svm_classifier.predict(X_test_vec)

train_time = time.time() - start_time
print(f"✅ 训练完成！耗时: {train_time:.2f}秒")

## 8. 步骤6：模型评估

对比三个模型的性能。

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 计算各项指标
models = {
    '朴素贝叶斯': y_pred_nb,
    '逻辑回归': y_pred_lr,
    'SVM': y_pred_svm
}

print("模型性能对比：")
print("="*80)
print(f"{'模型':<15} {'准确率':<10} {'精确率':<10} {'召回率':<10} {'F1值':<10}")
print("-"*80)

for model_name, y_pred in models.items():
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    
    print(f"{model_name:<15} {accuracy:<10.4f} {precision:<10.4f} {recall:<10.4f} {f1:<10.4f}")

### 8.1 详细分类报告

选择表现最好的模型，查看详细的分类报告。

In [ ]:
from sklearn.metrics import classification_report

# 使用逻辑回归的结果（通常效果最好）
print("逻辑回归模型详细报告：")
print("="*80)
print(classification_report(y_test, y_pred_lr, digits=4))

### 8.2 混淆矩阵

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# 计算混淆矩阵
cm = confusion_matrix(y_test, y_pred_lr)

# 获取类别名称
classes = sorted(set(y_test))

print("混淆矩阵：")
print("="*80)
print(f"{'':10s}", end="")
for cls in classes:
    print(f"{cls:10s}", end="")
print("  (预测)")

for i, cls in enumerate(classes):
    print(f"{cls:10s}", end="")
    for j in range(len(classes)):
        print(f"{cm[i][j]:10d}", end="")
    print()
print("(真实)")

## 9. 步骤7：模型应用

使用训练好的模型对新新闻进行分类。

In [ ]:
# 定义预测函数
def predict_news(text, model, vectorizer):
    """预测新闻类别"""
    # 分词
    words = jieba.cut(text)
    text_seg = " ".join(words)
    
    # 特征提取
    text_vec = vectorizer.transform([text_seg])
    
    # 预测
    prediction = model.predict(text_vec)[0]
    
    # 获取概率（如果模型支持）
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(text_vec)[0]
        return prediction, proba
    else:
        return prediction, None

print("✅ 预测函数定义完成！")

### 9.1 测试新闻分类

In [ ]:
# 准备测试新闻
test_news = [
    "华为发布最新5G手机，搭载自研芯片，性能强劲",
    "中国男篮在世界杯上战胜对手，成功晋级下一轮",
    "著名导演新片上映，首日票房破亿，观众好评如潮",
    "央行宣布降低存款准备金率，股市应声上涨",
    "某市发生交通事故，警方正在调查事故原因"
]

print("新闻分类测试：")
print("="*80)

for news in test_news:
    prediction, proba = predict_news(news, lr_classifier, vectorizer)
    
    print(f"\n新闻: {news}")
    print(f"预测类别: {prediction}")
    
    if proba is not None:
        print(f"置信度:")
        classes = lr_classifier.classes_
        for cls, prob in zip(classes, proba):
            print(f"  {cls}: {prob:.2%}")
    print("-"*80)

## 10. 步骤8：错误分析

分析模型预测错误的案例，找出改进方向。

In [ ]:
# 找出预测错误的样本
errors = []
for i, (true_label, pred_label) in enumerate(zip(y_test, y_pred_lr)):
    if true_label != pred_label:
        errors.append({
            'text': X_test[i],
            'true': true_label,
            'pred': pred_label
        })

print(f"错误样本数量: {len(errors)}")
print(f"错误率: {len(errors)/len(y_test):.2%}")
print(f"\n展示前5个错误案例：")
print("="*80)

for i, error in enumerate(errors[:5], 1):
    print(f"\n案例{i}:")
    print(f"文本: {error['text'][:100]}...")
    print(f"真实类别: {error['true']}")
    print(f"预测类别: {error['pred']}")
    print("-"*80)

## 11. 项目总结



### 11.1 我们完成了什么？

**1. 数据处理**
- 加载了50000条中文新闻数据
- 进行了分词预处理
- 划分了训练集和测试集

**2. 特征提取**
- 使用TF-IDF提取文本特征
- 将文本转换为数字向量

**3. 模型训练**
- 训练了3个不同的分类模型
- 对比了模型性能

**4. 模型评估**
- 使用多个指标评估模型
- 分析了混淆矩阵
- 进行了错误分析

**5. 模型应用**
- 实现了新闻分类预测功能
- 测试了实际应用效果

### 11.2 关键收获

**技术方面：**
- 掌握了完整的文本分类流程
- 学会了使用sklearn进行机器学习
- 理解了不同算法的特点

**实践经验：**
- 数据预处理很重要
- 特征提取影响模型效果
- 需要对比多个模型
- 错误分析帮助改进模型

**注意事项：**
- 数据质量比算法更重要
- 要在测试集上评估，不能用训练集
- 准确率不是唯一指标
- 实际应用中要持续优化

### 11.3 进一步优化方向

**1. 数据层面**
- 增加训练数据量
- 改进文本清洗方法
- 添加停用词过滤
- 使用更好的分词工具

**2. 特征层面**
- 尝试不同的n-gram组合
- 调整TF-IDF参数
- 尝试词向量（Word2Vec）

**3. 模型层面**
- 调整模型参数
- 尝试集成学习
- 尝试深度学习模型（BERT等）

**4. 应用层面**
- 添加置信度阈值
- 处理边界情况
- 定期更新模型

### 11.4 练习建议

**基础练习：**
1. 尝试调整TF-IDF的参数（max_features、min_df、max_df）
2. 添加停用词过滤，观察效果变化
3. 尝试使用bigram（2-gram）特征

**进阶练习：**
1. 实现一个简单的Web界面，输入新闻预测类别
2. 分析哪些词对分类最重要
3. 尝试处理多标签分类问题

**挑战练习：**
1. 收集自己的数据集，训练分类模型
2. 实现模型的保存和加载
3. 部署模型到实际应用中